# AgentAlign Lab — Eval-Only (SFT + DPO on Test Split)

Base model eval is already done (263 trajectories, 8.0% pass rate).
This notebook evaluates the **SFT and DPO adapters** on the same test tasks,
computes bootstrap CIs, and builds a failure taxonomy.

**Upload `outputs/gpu_eval_bundle.zip` as a Kaggle dataset.**
It contains the full code + both trained adapters. No git clone needed.

In [ ]:
# 0. GPU check
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 1. Extract bundle (code + adapters, no git clone needed)
import os, shutil
from pathlib import Path

work = Path('/kaggle/working/AgentAlign-Lab')
work.mkdir(exist_ok=True)

# Find the dataset — Kaggle auto-extracts zips
input_root = Path('/kaggle/input')
for d in input_root.iterdir():
    print(f'Dataset: {d.name}')
    for item in d.iterdir():
        print(f'  {item.name} (dir={item.is_dir()})')

# Copy everything from the dataset into the working dir
for d in input_root.iterdir():
    # Look for gpu_handoff subdir (Kaggle may have extracted into a subdir)
    for sub in d.iterdir():
        if sub.is_dir():
            # Check if this looks like the repo root (has src/ or pyproject.toml)
            if (sub / 'pyproject.toml').exists() or (sub / 'src').exists():
                print(f'Found repo root at {sub}')
                !cp -r {sub}/* /kaggle/working/AgentAlign-Lab/
                break
            # Could be gpu_handoff dir containing everything
            for subsub in sub.iterdir():
                if (subsub / 'pyproject.toml').exists() or subsub.name == 'src':
                    print(f'Found repo in {sub}')
                    !cp -r {sub}/* /kaggle/working/AgentAlign-Lab/
                    break

# Fallback: direct copy
if not (work / 'pyproject.toml').exists():
    for d in input_root.iterdir():
        !cp -r {d}/* /kaggle/working/AgentAlign-Lab/ 2>/dev/null || true
        for sub in d.iterdir():
            if sub.is_dir():
                !cp -r {sub}/* /kaggle/working/AgentAlign-Lab/ 2>/dev/null || true

os.chdir('/kaggle/working/AgentAlign-Lab')
!ls -la
print(f'\npyproject.toml exists: {Path("pyproject.toml").exists()}')
print(f'adapters exist: DPO={Path("outputs/adapters/qwen_dpo_final/adapter_config.json").exists()}, SFT={Path("outputs/adapters/qwen_sft_final/adapter_config.json").exists()}')

In [ ]:
# 2. Install
%cd /kaggle/working/AgentAlign-Lab
!pip install -e '.[ml,dashboard]' -q

In [ ]:
# 3. Verify tasks
import json, collections
for split in ['train', 'val', 'test']:
    tasks = [json.loads(l) for l in open(f'data/tasks/{split}.jsonl')]
    fam = collections.Counter(t['family'] for t in tasks)
    print(f'{split}: {len(tasks)} tasks')
    for k,v in sorted(fam.items()): print(f'  {k}: {v}')
    print()

# Verify adapters
from pathlib import Path
for name in ['qwen_dpo_final', 'qwen_sft_final']:
    p = Path(f'outputs/adapters/{name}/adapter_config.json')
    if p.exists():
        print(f'✓ {name}: {json.loads(p.read_text())["r"]}r LoRA')
    else:
        print(f'✗ {name}: NOT FOUND')

In [ ]:
# 4a. Run BASE model on test (4 reps)
!python scripts/02_run_agent.py \
    --split test --agent qwen_base \
    --model Qwen/Qwen2.5-Coder-1.5B-Instruct \
    --out runs/llm_test \
    --repetitions 4 \
    --temperature 0.8 --seed 42 --max-steps 8 \
    --device cuda --resume

In [ ]:
# 4b. Run SFT adapter on test (4 reps)
!python scripts/02_run_agent.py \
    --split test --agent qwen_sft \
    --model Qwen/Qwen2.5-Coder-1.5B-Instruct \
    --adapter outputs/adapters/qwen_sft_final \
    --out runs/llm_test \
    --repetitions 4 \
    --temperature 0.8 --seed 42 --max-steps 8 \
    --device cuda --resume

In [ ]:
# 4c. Run DPO adapter on test (4 reps)
!python scripts/02_run_agent.py \
    --split test --agent qwen_dpo \
    --model Qwen/Qwen2.5-Coder-1.5B-Instruct \
    --adapter outputs/adapters/qwen_dpo_final \
    --out runs/llm_test \
    --repetitions 4 \
    --temperature 0.8 --seed 42 --max-steps 8 \
    --device cuda --resume

In [ ]:
# 5. Score all test trajectories
!python scripts/03_score_trajectories.py \
    --runs-dir runs/llm_test \
    --out-dir data/trajectories/scored_llm_test \
    --clear

In [ ]:
# 6. Compute metrics + bootstrap CIs
import json, re, numpy as np
from pathlib import Path
from collections import defaultdict

scored_dir = Path('data/trajectories/scored_llm_test')
trajectories = []
for f in sorted(scored_dir.glob('*.jsonl')):
    for line in f.read_text().splitlines():
        if line.strip():
            trajectories.append(json.loads(line))

print(f'Total scored test trajectories: {len(trajectories)}')

by_agent = defaultdict(list)
for t in trajectories:
    by_agent[t.get('agent_id', 'unknown')].append(t)

for a in sorted(by_agent): print(f'  {a}: {len(by_agent[a])} trajectories')

def get_family(task_id):
    return re.sub(r'_\d+$', '', task_id) if task_id else 'unknown'

def compute_agent_metrics(trajs):
    total = len(trajs)
    passed = sum(1 for t in trajs if t.get('verifier_result', {}).get('passed', False))
    unsafe = sum(t.get('verifier_result', {}).get('unsafe_actions', 0) for t in trajs)
    failed_cmds = sum(t.get('verifier_result', {}).get('failed_commands', 0) for t in trajs)
    invalid = sum(t.get('verifier_result', {}).get('invalid_actions', 0) for t in trajs)
    avg_steps = float(np.mean([len(t.get('steps', [])) for t in trajs]))
    return {
        'total': total, 'passed': int(passed),
        'pass_rate': round(passed / max(total, 1), 4),
        'unsafe_actions': int(unsafe),
        'unsafe_rate': round(unsafe / max(total, 1), 4),
        'failed_commands': int(failed_cmds),
        'invalid_actions': int(invalid),
        'avg_steps': round(avg_steps, 2),
    }

results = {}
for agent, trajs in sorted(by_agent.items()):
    m = compute_agent_metrics(trajs)
    results[agent] = m
    print(f'\n{agent}: {m["passed"]}/{m["total"]} pass ({m["pass_rate"]*100:.1f}%), '
          f'unsafe={m["unsafe_actions"]}, invalid={m["invalid_actions"]}, '
          f'avg_steps={m["avg_steps"]}')

# Per-family breakdown
print('\n--- Per-family pass rates ---')
for agent in sorted(by_agent):
    trajs = by_agent[agent]
    family_stats = defaultdict(lambda: {'total': 0, 'passed': 0, 'unsafe': 0})
    for t in trajs:
        family = get_family(t.get('task_id', ''))
        family_stats[family]['total'] += 1
        if t.get('verifier_result', {}).get('passed', False):
            family_stats[family]['passed'] += 1
        family_stats[family]['unsafe'] += t.get('verifier_result', {}).get('unsafe_actions', 0)
    print(f'\n{agent}:')
    for fam in sorted(family_stats):
        s = family_stats[fam]
        rate = s['passed'] / max(s['total'], 1) * 100
        print(f'  {fam}: {s["passed"]}/{s["total"]} ({rate:.1f}%) unsafe={s["unsafe"]}')
    results[agent]['per_family'] = {k: dict(v) for k, v in family_stats.items()}

# Bootstrap CI
def paired_bootstrap_ci(a_trajs, b_trajs, n_boot=10000, seed=42):
    rng = np.random.RandomState(seed)
    a_by_task, b_by_task = defaultdict(list), defaultdict(list)
    for t in a_trajs:
        a_by_task[t['task_id']].append(1.0 if t.get('verifier_result', {}).get('passed', False) else 0.0)
    for t in b_trajs:
        b_by_task[t['task_id']].append(1.0 if t.get('verifier_result', {}).get('passed', False) else 0.0)
    common = sorted(set(a_by_task) & set(b_by_task))
    if not common: return 0, 0, 0, 1.0
    a_r = np.array([np.mean(a_by_task[t]) for t in common])
    b_r = np.array([np.mean(b_by_task[t]) for t in common])
    diffs = b_r - a_r
    obs = np.mean(diffs)
    n = len(diffs)
    boot = np.array([np.mean(diffs[rng.choice(n, n, replace=True)]) for _ in range(n_boot)])
    ci_lo, ci_hi = float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))
    p = float(np.mean(np.abs(boot - boot.mean()) >= abs(obs)))
    return round(obs, 4), round(ci_lo, 4), round(ci_hi, 4), round(p, 4)

print('\n--- Bootstrap 95% CIs (10k resamples) ---')
for key, a, b, label in [
    ('dpo_vs_base', 'qwen_base', 'qwen_dpo', 'DPO - Base'),
    ('sft_vs_base', 'qwen_base', 'qwen_sft', 'SFT - Base'),
    ('dpo_vs_sft', 'qwen_sft', 'qwen_dpo', 'DPO - SFT'),
]:
    if a in by_agent and b in by_agent:
        diff, lo, hi, p = paired_bootstrap_ci(by_agent[a], by_agent[b])
        spans = 'YES → no detectable improvement' if lo <= 0 <= hi else 'NO'
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
        print(f'{label}: {diff:+.4f} [{lo:+.4f}, {hi:+.4f}] p={p} {sig}')
        print(f'  CI spans zero: {spans}')
        results[key] = {'diff': diff, 'ci_lo': lo, 'ci_hi': hi, 'p_value': p}
    else:
        print(f'{label}: SKIPPED (missing agent)')

Path('outputs/evals').mkdir(parents=True, exist_ok=True)
Path('outputs/evals/eval_results.json').write_text(json.dumps(results, indent=2))
print('\n✅ Saved to outputs/evals/eval_results.json')

In [ ]:
# 7. Failure taxonomy — 10 DPO test failures
import json, re
from pathlib import Path
from collections import Counter

scored_dir = Path('data/trajectories/scored_llm_test')
failures = []
for f in sorted(scored_dir.glob('*.jsonl')):
    for line in f.read_text().splitlines():
        if not line.strip(): continue
        t = json.loads(line)
        if t.get('agent_id') == 'qwen_dpo' and not t.get('verifier_result', {}).get('passed', False):
            failures.append(t)

print(f'Total DPO failures on test: {len(failures)}')

def classify(t):
    vr = t.get('verifier_result', {})
    steps = t.get('steps', [])
    n = len(steps)
    if vr.get('unsafe_actions', 0) > 0: return 'unsafe_action'
    if vr.get('invalid_actions', 0) > n * 0.5 and n > 0: return 'mostly_invalid'
    if n >= 7: return 'max_steps_exhausted'
    if vr.get('failed_commands', 0) > 0 and n <= 3: return 'early_cmd_failure'
    if n <= 2: return 'premature_stop'
    return 'wrong_output'

sample = failures[:10]
taxonomy = []
for i, f in enumerate(sample):
    vr = f.get('verifier_result', {})
    entry = {
        'task_id': f.get('task_id'), 'family': re.sub(r'_\d+$', '', f.get('task_id', '')),
        'category': classify(f), 'n_steps': len(f.get('steps', [])),
        'unsafe': vr.get('unsafe_actions', 0), 'invalid': vr.get('invalid_actions', 0),
        'failed_cmds': vr.get('failed_commands', 0), 'score': vr.get('score', 0),
    }
    taxonomy.append(entry)
    print(f'{i+1}. {entry["task_id"]}: {entry["category"]} (steps={entry["n_steps"]}, unsafe={entry["unsafe"]}, invalid={entry["invalid"]})')

all_labels = [classify(f) for f in failures]
print(f'\nFull distribution ({len(failures)} failures):')
for label, count in Counter(all_labels).most_common():
    print(f'  {label}: {count} ({count/len(failures)*100:.0f}%)')

out = {'sample': taxonomy, 'total': len(failures), 'distribution': dict(Counter(all_labels).most_common())}
Path('outputs/evals/failure_taxonomy.json').write_text(json.dumps(out, indent=2))
print('\n✅ Saved to outputs/evals/failure_taxonomy.json')

In [ ]:
# 8. Package for download
import shutil
from pathlib import Path

export = Path('/kaggle/working/export')
if export.exists(): shutil.rmtree(export)
export.mkdir()

for src in ['outputs/evals', 'data/trajectories/scored_llm_test', 'runs/llm_test']:
    s = Path(src)
    if not s.exists():
        print(f'  SKIP: {src}')
        continue
    d = export / src
    if s.is_dir():
        if d.exists(): shutil.rmtree(d)
        shutil.copytree(s, d)
    else:
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s, d)
    print(f'  ✓ {src}')

shutil.make_archive('/kaggle/working/agentalign_eval_results', 'zip', export)
print(f'\n✅ Download: /kaggle/working/agentalign_eval_results.zip')
print(f'Then run locally:')
print(f'  python scripts/10_import_colab_artifacts.py --zip-path ~/Downloads/agentalign_eval_results.zip --force')